# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not treat as dict; use attributes

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their @id, name, and fields
print('Record sets overview:')
record_sets = list(dataset.record_sets)

if not record_sets:
    print('No record sets found. Inspecting croissant distributions for possible file-level record sets.')
    # Some datasets describe record sets within distributions.
    for distribution in getattr(metadata, 'distribution', []):
        print(f"Distribution @id: {getattr(distribution, '@id', None)}")
    print('Please consult the Croissant schema in detail for nested records.')
else:
    for record_set in record_sets:
        print(f"- @id: {record_set['@id']} | name: {record_set.get('name', '')}")
        fields = record_set.get('fields', [])
        print('  Fields:')
        for field in fields:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '')}")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.
All entities (record sets, fields, columns) are referred to by their `@id`.

In [ ]:
# Step 1: List all available record set @id's
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# If no record sets are listed, print a message and skip data extraction
if not all_record_set_ids:
    print("No record sets detected in the Croissant schema.")
    dataframes = {}
else:
    print("Record set @id's:")
    for rsid in all_record_set_ids:
        print(f"- {rsid}")

    # Step 2: Load records from each record set into a DataFrame
    dataframes = {}
    for record_set_id in all_record_set_ids:
        print(f"\nLoading record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
            # Display the first few rows
            display(df.head())
        except Exception as e:
            print(f"Failed to load records for {record_set_id}: {e}")

# For demonstration, pick the first record set (if any) for further analysis
if dataframes:
    primary_rs_id = list(dataframes.keys())[0]
    print(f"Using record set for EDA: {primary_rs_id}")
else:
    primary_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Perform analysis of numeric fields and group by categorical fields, referencing fields using their `@id`.
Typical steps: filtering records, normalization, grouping.

In [ ]:
import numpy as np

if primary_rs_id and not dataframes[primary_rs_id].empty:
    df = dataframes[primary_rs_id]

    # Identify numeric fields by checking dtypes
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        print("No numeric fields found for EDA.")
    else:
        # Select the first numeric field for demonstration, using its @id
        numeric_field_id = numeric_cols[0]
        print(f"Analyzing numeric field: {numeric_field_id}")

        # Example: filter values > threshold (arbitrary 10th percentile)
        threshold = df[numeric_field_id].quantile(0.10)
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (10th percentile):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a categorical/grouping field
        candidate_group_cols = df.select_dtypes(include=['object', 'category']).columns
        group_field_id = None
        for candidate in candidate_group_cols:
            if df[candidate].nunique() > 1 and df[candidate].nunique() < 20:
                group_field_id = candidate
                break
        if group_field_id:
            print(f"Grouping data by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped mean values:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No records/data found for EDA.")

## 5. Visualization
Visualize distributions or relationships between dataset fields.
Use only available data and field `@id`s for labeling axes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id and not dataframes[primary_rs_id].empty:
    df = dataframes[primary_rs_id]
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        field_id = numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field_id], kde=True)
        plt.title(f"Distribution of {field_id}")
        plt.xlabel(field_id)
        plt.ylabel("Count")
        plt.show()
        
        # If a group field exists
        candidate_group_cols = df.select_dtypes(include=['object', 'category']).columns
        group_field_id = None
        for candidate in candidate_group_cols:
            if df[candidate].nunique() > 1 and df[candidate].nunique() < 10:
                group_field_id = candidate
                break
        if group_field_id:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[group_field_id], y=df[field_id])
            plt.title(f"{field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(field_id)
            plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a rich regression and demographic dataset described using the Croissant standard with `mlcroissant`.

We accessed metadata, programmatically listed record sets and field `@id`s, loaded tabular data, conducted exploratory analysis, normalized numeric variables, and summarized group-level patterns referencing entities via their unique `@id`s throughout.

This approach is extensible to any dataset with a Croissant schema, promoting reproducible and FAIR data science workflows.